<a href="https://colab.research.google.com/github/urbdaniel86/DMEyF/blob/main/competencia_01/z402_Feature_Engineering_en_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Feature Engineering en SQL

A continuación, veremos cómo calcular diferentes variables para el feature engineering utilizando SQL.


In [ ]:
%pip install jupysql
%pip install duckdb-engine

In [ ]:
import duckdb
import pandas as pd

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

%sql duckdb://

In [ ]:
dataset_path = '/content/drive/MyDrive/DMEyF/2026/notebooks/data/'
dataset_file = 'competencia_01.csv'

In [ ]:
%%sql
create or replace table competencia_01 as
select
    *
from read_csv_auto("{{dataset_path + dataset_file}}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success


In [ ]:
%%sql
select
    Master_Fvencimiento
    , Visa_Fvencimiento
    , greatest(Master_Fvencimiento, Visa_Fvencimiento) as tc_fvencimiento_mayor
    , least(Master_Fvencimiento, Visa_Fvencimiento) as tc_fvencimiento_menor
from competencia_01 limit 10

,Master_Fvencimiento,Visa_Fvencimiento,tc_fvencimiento_mayor,tc_fvencimiento_menor
0,<NA>,-182,-182,-182
1,<NA>,-152,-152,-152
2,<NA>,-121,-121,-121
3,<NA>,-91,-91,-91
4,<NA>,-60,-60,-60
5,<NA>,-2220,-2220,-2220
6,-943,<NA>,-943,-943
7,-913,-1825,-913,-1825
8,<NA>,<NA>,<NA>,<NA>
9,<NA>,<NA>,<NA>,<NA>


Lo siguiente es querer operar dos variables, como por ejemplo sumarla. Esto es sencillo


In [ ]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , Master_msaldototal + Visa_msaldototal as tc_saldo_total
from competencia_01 limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,0.00,NaN
1,NaN,0.00,NaN
2,NaN,0.00,NaN
3,NaN,0.00,NaN
4,NaN,0.00,NaN
5,NaN,0.00,NaN
6,113086.11,NaN,NaN
7,126857.66,-5196.97,121660.69
8,NaN,NaN,NaN
9,NaN,NaN,NaN


Pero un DS de a de veras mirará los datos y se encontrará con un campo que es null cuando se lo suma a otro dará null.

In [ ]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , Master_msaldototal + Visa_msaldototal as tc_saldo_total
from competencia_01 where Master_msaldototal is null limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,0.0,NaN
1,NaN,0.0,NaN
2,NaN,0.0,NaN
3,NaN,0.0,NaN
4,NaN,0.0,NaN
5,NaN,0.0,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


Esto no siempre es deseable y puede ser fácilmente evitable

In [ ]:
%%sql
select
    Master_msaldototal
    , Visa_msaldototal
    , ifnull(Master_msaldototal, 0) + ifnull(Visa_msaldototal, 0) as tc_saldo_total
from competencia_01 limit 10

,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,0.00,0.00
1,NaN,0.00,0.00
2,NaN,0.00,0.00
3,NaN,0.00,0.00
4,NaN,0.00,0.00
5,NaN,0.00,0.00
6,113086.11,NaN,113086.11
7,126857.66,-5196.97,121660.69
8,NaN,NaN,0.00
9,NaN,NaN,0.00


In [ ]:
%%sql
CREATE OR REPLACE MACRO suma_sin_null(a, b) AS ifnull(a, 0) + ifnull(b, 0);


,Success


In [ ]:
%%sql
select distinct
    Master_msaldototal
    , Visa_msaldototal
    , suma_sin_null(Master_msaldototal, Visa_msaldototal) as tc_saldo_total
from competencia_01 where Master_msaldototal is null limit 10


,Master_msaldototal,Visa_msaldototal,tc_saldo_total
0,NaN,8400.64,8400.64
1,NaN,3934.59,3934.59
2,NaN,592.00,592.00
3,NaN,NaN,0.00
4,NaN,244935.25,244935.25
5,NaN,3833.97,3833.97
6,NaN,6180.97,6180.97
7,NaN,3067.86,3067.86
8,NaN,20455.10,20455.10
9,NaN,19620.49,19620.49


TAREA: Escriba una macro para hacer un ratio de dos variables que sea seguro, donde no solo hay campos con null, también esta el problema de la división por cero. Como es costumbre comparta su solución por este canal. Lea https://duckdb.org/docs/sql/functions/numeric.html para referencias de funciones que puede usar.

---

"Claro!" me dirá, mientras lee esto con un mate en la mano, "para cosas fáciles usar SQL alcanza, pero para algo más complicado como crear campos contra el data drifting es difícil".... elija su medicina:

In [ ]:
%%sql
select
    foto_mes
    , numero_de_cliente
    , cliente_antiguedad
    , row_number() over (partition by numero_de_cliente order by foto_mes) as cliente_antiguedad_2
    , percent_rank() over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_3
    , cume_dist() over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_4
    , ntile(4) over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_5
    , ntile(10) over (partition by foto_mes order by cliente_antiguedad) as cliente_antiguedad_6
from competencia_01
order by numero_de_cliente, cliente_antiguedad


,foto_mes,numero_de_cliente,cliente_antiguedad,cliente_antiguedad_2,cliente_antiguedad_3,cliente_antiguedad_4,cliente_antiguedad_5,cliente_antiguedad_6
0,202103,12159854,134,1,0.545491,0.550712,3,6
1,202104,12159854,135,2,0.547454,0.552706,3,6
2,202105,12159854,136,3,0.549647,0.554852,3,6
3,202106,12159854,137,4,0.551510,0.556704,3,6
4,202107,12159854,138,5,0.553128,0.558303,3,6
...,...,...,...,...,...,...,...,...
983056,202108,78249586,1,1,0.000000,0.001585,1,1
983057,202108,78249850,1,1,0.000000,0.001585,1,1
983058,202108,78250419,1,1,0.000000,0.001585,1,1
983059,202108,78253625,1,1,0.000000,0.001585,1,1


Qué paso? use las hermosas funciones analíticas de SQL. Al campo cliente_antiguedad (que no sufre de data drifting, solo esta para dar el ejemplo) para cada período (partition by foto_mes) la ordeno (order by cliente_antiguedad) y luego calculo las métricas de orden que pueden encontrar acá https://duckdb.org/docs/sql/window_functions.html#general-purpose-window-functions.

Seguiremos usando las funciones analíticas de SQL, esta vez para calcular features que utilizan valores del pasado.

Qué pasa si quiero agregar un feature que muestre el valor del periodo anterior?


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
from competencia_01
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter
0,12162107,202103,58,<NA>
1,12162107,202104,40,58
2,12162107,202105,32,40
3,12162107,202106,18,32
4,12162107,202107,14,18
5,12162107,202108,14,14
6,12163274,202103,116,<NA>
7,12163274,202104,129,116
8,12163274,202105,125,129
9,12163274,202106,129,125


Podemos calcular el delta (diferencia) entre el valor pasado y el presente, para uno o varios meses


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
  , ctrx_quarter - lag_1_ctrx_quarter as delta_1_ctrx_quarter
  , ctrx_quarter - lag(ctrx_quarter, 2) over (partition by numero_de_cliente order by foto_mes) as lag_2_ctrx_quarter
from competencia_01
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter,delta_1_ctrx_quarter,lag_2_ctrx_quarter
0,12162107,202103,58,<NA>,<NA>,<NA>
1,12162107,202104,40,58,-18,<NA>
2,12162107,202105,32,40,-8,-26
3,12162107,202106,18,32,-14,-22
4,12162107,202107,14,18,-4,-18
5,12162107,202108,14,14,0,-4
6,12163274,202103,116,<NA>,<NA>,<NA>
7,12163274,202104,129,116,13,<NA>
8,12163274,202105,125,129,-4,9
9,12163274,202106,129,125,4,0


Si necesitamos ya no solo traer un valor del pasado, sino una secuencia de valores, por ejemplo para calcular la media móvil con los últimos 3 meses anteriores? se puede hacer fácilmente


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , lag(ctrx_quarter, 1) over (partition by numero_de_cliente order by foto_mes) as lag_1_ctrx_quarter
  , lag(ctrx_quarter, 2) over (partition by numero_de_cliente order by foto_mes) as lag_2_ctrx_quarter
  , lag(ctrx_quarter, 3) over (partition by numero_de_cliente order by foto_mes) as lag_3_ctrx_quarter
  , avg(ctrx_quarter) over (partition by numero_de_cliente
                            order by foto_mes
                            rows between 3 preceding and current row) as avg_3_ctrx_quarter
from competencia_01
order by numero_de_cliente, foto_mes desc
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,lag_1_ctrx_quarter,lag_2_ctrx_quarter,lag_3_ctrx_quarter,avg_3_ctrx_quarter
0,12159854,202108,43,45,44,54,46.500000
1,12159854,202107,45,44,54,59,50.500000
2,12159854,202106,44,54,59,64,55.250000
3,12159854,202105,54,59,64,<NA>,59.000000
4,12159854,202104,59,64,<NA>,<NA>,61.500000
5,12159854,202103,64,<NA>,<NA>,<NA>,64.000000
6,12159858,202108,70,68,60,64,65.500000
7,12159858,202107,68,60,64,71,65.750000
8,12159858,202106,60,64,71,65,65.000000
9,12159858,202105,64,71,65,<NA>,66.666667


Si embargo puede resultar incómodo escribir constantemente el over partition sobre todo si se buscan aplicar muchas veces para distintas funciones. Para reducir el código se puede usar la siguiente sintaxis



In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  , avg(ctrx_quarter) over ventana_3 as ctrx_quarter_media_3
  , max(ctrx_quarter) over ventana_3 as ctrx_quarter_max_3
  , min(ctrx_quarter) over ventana_3 as ctrx_quarter_min_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,ctrx_quarter_media_3,ctrx_quarter_max_3,ctrx_quarter_min_3
0,12162107,202103,58,58.000000,58,58
1,12162107,202104,40,49.000000,58,40
2,12162107,202105,32,43.333333,58,32
3,12162107,202106,18,37.000000,58,18
4,12162107,202107,14,26.000000,40,14
5,12162107,202108,14,19.500000,32,14
6,12163274,202103,116,116.000000,116,116
7,12163274,202104,129,122.500000,129,116
8,12163274,202105,125,123.333333,129,116
9,12163274,202106,129,124.750000,129,116


Para saber más que funciones tenemos disponibles, recomiendo ver los siguientes links:

https://duckdb.org/docs/archive/0.8.1/sql/window_functions
https://duckdb.org/docs/archive/0.8.1/sql/aggregates
Un caso más, que ni me voy a molestar en explicar que significa...


In [ ]:
%%sql
select
  numero_de_cliente
  , foto_mes
  , ctrx_quarter
  ,regr_slope(ctrx_quarter, cliente_antiguedad) over ventana_3 as ctrx_quarter_slope_3
from competencia_01
window ventana_3 as (partition by numero_de_cliente order by foto_mes rows between 3 preceding and current row)
limit 10


,numero_de_cliente,foto_mes,ctrx_quarter,ctrx_quarter_slope_3
0,12173792,202103,175,NaN
1,12173792,202104,190,15.0
2,12173792,202105,213,19.0
3,12173792,202106,237,20.9
4,12173792,202107,250,20.4
5,12173792,202108,285,22.9
6,12182374,202103,144,NaN
7,12182374,202104,155,11.0
8,12182374,202105,162,9.0
9,12182374,202106,154,3.7


... Alguno dirá "tenemos que escribir todo esto a mano? Son muchas variables!". Bueno no, use los conocimientos de programación para que la computadora trabaje para usted. Si tenemos una lista de campos


In [ ]:
campos = ['active_quarter', 'cliente_vip', 'internet', 'cliente_edad', 'cliente_antiguedad', 'mrentabilidad']


Podemos hacer un script muy sencillo que nos genere el texto que hay que poner en una query para generar esas variables


In [ ]:
nuevos_features = ""
for campo in campos:
  nuevos_features += f"\n, regr_slope({campo}, cliente_antiguedad) over ventana_3 as ctrx_{campo}_slope_3"
print(nuevos_features)



, regr_slope(active_quarter, cliente_antiguedad) over ventana_3 as ctrx_active_quarter_slope_3
, regr_slope(cliente_vip, cliente_antiguedad) over ventana_3 as ctrx_cliente_vip_slope_3
, regr_slope(internet, cliente_antiguedad) over ventana_3 as ctrx_internet_slope_3
, regr_slope(cliente_edad, cliente_antiguedad) over ventana_3 as ctrx_cliente_edad_slope_3
, regr_slope(cliente_antiguedad, cliente_antiguedad) over ventana_3 as ctrx_cliente_antiguedad_slope_3
, regr_slope(mrentabilidad, cliente_antiguedad) over ventana_3 as ctrx_mrentabilidad_slope_3





Con la salida de esa celda, arme la query agregando las nuevas líneas y la ejecuta.

Lo que acabamos de hacer de manera muy simple es como "funcionan" sistemas como **dbt** que están tan de moda en el mundo de los datos.

La última reflexión, la creación de nuevas features es un proceso computacionalmente rápido pero intenso. Si ejecutó lo anterior pudo haber visto que en poco minutos tenía sus nuevas variables. Pero, también pudo haberle fallado por temas de recursos. Miles de variables necesitan los recursos adecuados. Use la nube, una máquina grande, al menos que sepa bien como optimizar las queries.


Y a no olvidarse guardar las nueva tabla

In [ ]:
%%sql
COPY competencia_01 TO '{dataset_path}competencia_01_fe.csv' (FORMAT CSV, HEADER TRUE);


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,Success
